# Cañadata — exploración de datos

Este notebook es de **lectura**: lo recorres antes de la primera sesión para anclarte en el dataset que usaremos durante todo el taller.

Si tu memoria de EDA está oxidada, no pasa nada — la idea aquí no es enseñarte EDA otra vez, sino que llegues a S1 sabiendo cómo se ven los datos de Cañadata.

Si te entra el gusanillo, al final hay retos opcionales marcados con 🚀.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Encuentra la raíz del proyecto, ya estés en pre_class/ o en el root
ROOT = Path.cwd()
if ROOT.name == "pre_class":
    ROOT = ROOT.parent

df = pd.read_csv(ROOT / "data" / "canadata_leads.csv")
print(f"{len(df)} leads, {len(df.columns)} columnas")
df.head()


## ¿Qué hay en cada columna?

Identidad: `lead_id`, `company_name`, `industry`, `company_size`, `country`, `signup_date`.

Embudo: `source`, `demo_requested`, `emails_opened`, `response_time_hours`, `n_meetings`, `decision_maker_contacted`, `quoted_acv_eur`.

Texto libre: `company_description` (1-2 frases en español por lead, lo que verá el LLM en zero-shot).

Targets: `converted` (binario), `converted_within_days` (numérico, NULL si no convirtió), `quoted_acv_eur` (también es target de regresión).

Verdad oculta: `lead_segment_truth` (3 arquetipos plantados — solo para evaluar clustering, NUNCA como feature).

**Aviso**: el dataset es sintético pero está sucio a propósito. Como en cualquier CRM real, vas a encontrar duplicados, NaNs, mayúsculas inconsistentes (`SaaS` vs `saas` vs `SAAS`), alias de país (`Spain` / `España` / `ESP`), tipos mezclados (booleanos como string), outliers y leads de prueba. Búscalos.


In [ ]:
df.dtypes


In [ ]:
df.describe(include="all")


## Distribuciones rápidas

Mira por encima: cómo se distribuyen los planes, los canales, los países, y la conversión.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
df["source"].value_counts().plot(kind="bar", ax=axes[0, 0], title="source")
df["industry"].value_counts().plot(kind="bar", ax=axes[0, 1], title="industry")
df["country"].value_counts().plot(kind="bar", ax=axes[1, 0], title="country")
df["converted"].value_counts().plot(kind="bar", ax=axes[1, 1], title="converted")
plt.tight_layout()
plt.show()


## Cómo varía la conversión

Esto es la pregunta de negocio principal: ¿qué hace que un lead convierta?


In [ ]:
by_source = df.groupby("source")["converted"].mean().sort_values(ascending=False)
by_industry = df.groupby("industry")["converted"].mean().sort_values(ascending=False)
print("Tasa de conversión por source:")
print(by_source.round(3).to_string())
print("\nTasa de conversión por industria:")
print(by_industry.round(3).to_string())


## Distribución de tamaño y ACV

`company_size` y `quoted_acv_eur` son log-normales. Visualízalos en escala log para que se entiendan.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(np.log10(df["company_size"]), bins=30, color="steelblue")
axes[0].set_title("log10(company_size)")
axes[1].hist(np.log10(df["quoted_acv_eur"]), bins=30, color="darkorange")
axes[1].set_title("log10(quoted_acv_eur)")
plt.tight_layout()
plt.show()


## Estructura temporal

Los `signup_date` cubren 24 meses con tendencia y estacionalidad plantadas. Veámoslo.


In [ ]:
df["signup_date"] = pd.to_datetime(df["signup_date"])
monthly = df.set_index("signup_date").resample("MS").size()
monthly_conv = df[df["converted"]].set_index("signup_date").resample("MS").size().reindex(monthly.index, fill_value=0)

fig, ax = plt.subplots(figsize=(11, 4))
monthly.plot(ax=ax, label="leads totales", marker="o")
monthly_conv.plot(ax=ax, label="leads convertidos", marker="s")
ax.set_title("Volumen mensual")
ax.legend()
plt.tight_layout()
plt.show()


## Echa un vistazo a las descripciones

Esto es lo único que verá el LLM en modo zero-shot durante S2. Lee 5-10 ejemplos para hacerte una idea de qué señal hay (y qué señal NO hay) en el texto.


In [ ]:
for _, row in df.sample(8, random_state=0).iterrows():
    print(f"[{row['converted']}] {row['company_description']}")


## La verdad oculta

El generador plantó tres arquetipos. Están guardados en `lead_segment_truth`. **No** los uses como feature en clasificación o regresión — sí los usaremos para evaluar la calidad del clustering en S1.


In [ ]:
df["lead_segment_truth"].value_counts()


In [ ]:
df.groupby("lead_segment_truth")[["company_size", "n_meetings", "response_time_hours", "quoted_acv_eur", "converted"]].mean().round(2)


## Encuentra el desorden

Antes de modelar nada, hay que limpiar. Esto te lo encuentras en cualquier dataset real. Algunas líneas para pillar la idea — si ves muchas variantes raras, no estás loco, están plantadas.


In [ ]:
print('industry — value_counts (con NaN):')
print(df['industry'].value_counts(dropna=False).head(15))
print()
print('country — value_counts (con NaN):')
print(df['country'].value_counts(dropna=False).head(15))
print()
print('source — value_counts:')
print(df['source'].value_counts(dropna=False).head(15))


In [ ]:
print('Tipos por columna:')
print(df.dtypes.to_string())
print()
print('demo_requested — primeros 10 valores únicos:')
print(list(df['demo_requested'].unique()[:10]))
print()
print('Filas duplicadas exactas:', df.duplicated().sum())
print('lead_id duplicados:      ', df['lead_id'].duplicated().sum())
print()
print('NaN por columna:')
print(df.isna().sum().sort_values(ascending=False).to_string())


## 🚀 Si te quedas con ganas

1. Encuentra los leads de prueba (`TEST LEAD - DELETE`, `asdfasdf`, etc.). ¿Cuántos hay y dónde están?
2. Calcula el lifetime value medio por arquetipo (suponiendo `quoted_acv_eur` como anual). ¿Quién es realmente el más rentable?
3. ¿Hay diferencias claras de `response_time_hours` por `country`? ¿Por `source`? Cuidado con los outliers (`5000h`).
4. Construye un heatmap `industry × country` con la tasa de conversión. ¿Hay celdas vacías?
5. Mira `converted_within_days` solo en convertidos. ¿La distribución se parece más a una exponencial o a una log-normal? Pista: histograma + log-scale.
6. Cuenta cuántas descripciones están en inglés. ¿Cómo lo detectarías sin leer cada una?
7. Encuentra los `signup_date` futuros. ¿Qué política aplicarías?
